# Fairness, Bias & Explainability Analysis: CIFAR-10 MobileNetV2

**Model analysed:** `cifar10_mobilenetv2.keras` (frozen MobileNetV2 + dense head, ~81.8% test accuracy).

**Important framing.** CIFAR-10 has *no demographic attributes*, so a classic fairness audit (gender, race, age) is not possible on this dataset. This notebook therefore audits two things that *are* measurable:

1. **Class-level disparity:** does the model serve some classes much worse than others (recall, false-positive rate, calibration)?
2. **Condition-slice disparity:** does accuracy drop for images that are dark, low-contrast, desaturated, noisy or blurred? These are stand-ins for "groups" and mirror how subgroup audits work on real data.

Explainability uses permutation/occlusion importance (global) plus **SHAP** and **LIME** (local), including on misclassified examples.

Run top to bottom (`Kernel > Restart & Run All`). Allow roughly 10-20 minutes on CPU; the SHAP aggregation is the slowest step. Final cell writes `fairness_results.md` with all numbers.

In [ ]:
# !pip install -r requirements.txt
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from scipy.ndimage import gaussian_filter
from sklearn.metrics import confusion_matrix, classification_report
from skimage.segmentation import slic, mark_boundaries
import shap
from lime import lime_image

np.random.seed(0); tf.random.set_seed(0)
pd.set_option('display.width', 160)

MODEL_PATH = 'cifar10_mobilenetv2.keras'
class_names = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
RESULTS = {}   # collected for the auto-generated report at the end

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train, y_test = y_train.flatten(), y_test.flatten()

def predict(x):
    return model.predict(np.asarray(x, dtype=np.float32), batch_size=256, verbose=0)

probs  = predict(x_test)
y_pred = probs.argmax(1)
conf   = probs.max(1)
correct = y_pred == y_test
print(f'Overall test accuracy: {correct.mean()*100:.2f}%')
RESULTS['overall_accuracy'] = float(correct.mean())

## 1. Class-level bias: per-class performance and calibration

Metrics per class: **recall** (true-positive rate), **false-positive rate**, **precision**, and **calibration** (does confidence match accuracy?). Disparity is summarised by the recall gap (max - min) and the min/max recall ratio (analogous to the "80% rule").

In [ ]:
def ece(conf, correct, bins=10):
    edges = np.linspace(0, 1, bins+1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return e

cm = confusion_matrix(y_test, y_pred)
tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tn = cm.sum() - tp - fp - fn
rep = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)

class_df = pd.DataFrame({
    'precision': [rep[c]['precision'] for c in class_names],
    'recall (TPR)': tp / (tp + fn),
    'false-pos rate': fp / (fp + tn),
    'ECE (calibration)': [ece(conf[y_test == i], correct[y_test == i]) for i in range(10)],
}, index=class_names).round(4)
display(class_df)

rec = class_df['recall (TPR)']
RESULTS['recall_gap'] = float(rec.max() - rec.min())
RESULTS['recall_ratio_min_max'] = float(rec.min() / rec.max())
RESULTS['worst_class'], RESULTS['best_class'] = rec.idxmin(), rec.idxmax()
print(f"Recall gap (max-min): {RESULTS['recall_gap']:.3f} | min/max ratio: {RESULTS['recall_ratio_min_max']:.3f}")
print(f"Worst class: {RESULTS['worst_class']} ({rec.min():.3f}) | Best: {RESULTS['best_class']} ({rec.max():.3f})")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(class_names, rec, color='steelblue'); ax[0].axhline(rec.mean(), color='red', ls='--', label='mean')
ax[0].set_title('Per-class recall'); ax[0].tick_params(axis='x', rotation=45); ax[0].legend()
ax[1].bar(class_names, class_df['ECE (calibration)'], color='darkorange')
ax[1].set_title('Per-class calibration error (ECE, lower is better)'); ax[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# Which mistakes dominate? (row = true class, column = predicted class)
off = cm.copy(); np.fill_diagonal(off, 0)
pairs = sorted([(off[i, j], class_names[i], class_names[j]) for i in range(10) for j in range(10) if i != j], reverse=True)[:8]
pairs_df = pd.DataFrame(pairs, columns=['count', 'true', 'predicted'])
display(pairs_df)
RESULTS['top_confusions'] = pairs_df.to_dict('records')

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(cm, cmap='Blues'); ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xticklabels(class_names, rotation=45, ha='right'); ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Confusion matrix (counts)')
for i in range(10):
    for j in range(10): ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=7, color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.tight_layout(); plt.show()

## 2. Condition-slice bias: brightness, contrast, saturation

Test images are split into terciles (low / mid / high) on each property. If accuracy differs sharply between slices, the model under-serves some conditions. 95% Wilson confidence intervals show whether gaps are larger than sampling noise.

In [ ]:
xf = x_test.astype(np.float32) / 255
brightness = xf.mean(axis=(1, 2, 3))
contrast   = xf.std(axis=(1, 2, 3))
mx, mn = xf.max(3), xf.min(3)
saturation = (np.where(mx > 0, (mx - mn) / np.maximum(mx, 1e-6), 0)).mean(axis=(1, 2))

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z*z/n; c = p + z*z/(2*n); r = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return (c - r)/d, (c + r)/d

rows = []
for name, v in [('brightness', brightness), ('contrast', contrast), ('saturation', saturation)]:
    bins = pd.qcut(v, 3, labels=['low', 'mid', 'high'])
    for b in ['low', 'mid', 'high']:
        m = np.asarray(bins == b); k, n = correct[m].sum(), m.sum()
        lo, hi = wilson(k, n)
        rows.append({'property': name, 'slice': b, 'n': int(n), 'accuracy': k/n, 'CI low': lo, 'CI high': hi})
slice_df = pd.DataFrame(rows).round(4)
display(slice_df)

gaps = slice_df.groupby('property')['accuracy'].agg(lambda s: s.max() - s.min())
RESULTS['slice_gaps'] = gaps.round(4).to_dict()
print('Accuracy gap across slices:'); print(gaps.round(4))

fig, ax = plt.subplots(figsize=(7, 4))
for i, (name, g) in enumerate(slice_df.groupby('property')):
    ax.errorbar(np.arange(3) + i*0.15, g['accuracy'], yerr=[g['accuracy'] - g['CI low'], g['CI high'] - g['accuracy']], fmt='o-', capsize=3, label=name)
ax.set_xticks(range(3)); ax.set_xticklabels(['low', 'mid', 'high']); ax.set_ylabel('Accuracy'); ax.legend(); ax.set_title('Accuracy by image-condition slice (95% CI)')
plt.tight_layout(); plt.show()

In [ ]:
# Class x brightness heat map: is a class especially fragile in dark / bright images?
bins_b = pd.qcut(brightness, 3, labels=['low', 'mid', 'high'])
heat = np.array([[correct[(y_test == c) & np.asarray(bins_b == b)].mean() for b in ['low', 'mid', 'high']] for c in range(10)])
fig, ax = plt.subplots(figsize=(4.5, 5))
im = ax.imshow(heat, cmap='RdYlGn', vmin=0.4, vmax=1); ax.set_yticks(range(10)); ax.set_yticklabels(class_names)
ax.set_xticks(range(3)); ax.set_xticklabels(['dark', 'mid', 'bright']); ax.set_title('Per-class accuracy by brightness')
for i in range(10):
    for j in range(3): ax.text(j, i, f'{heat[i, j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im); plt.tight_layout(); plt.show()

## 3. Robustness slices: how does each class degrade under perturbation?

A fair model should degrade gracefully for *all* classes. A large worst-class drop signals a hidden disparity.

In [ ]:
rng = np.random.default_rng(0)
perturb = {
    'clean': lambda x: x,
    'darker (x0.5)': lambda x: x * 0.5,
    'brighter (+60)': lambda x: np.clip(x + 60, 0, 255),
    'gaussian noise (sd=20)': lambda x: np.clip(x + rng.normal(0, 20, x.shape), 0, 255),
    'blur (sigma=1)': lambda x: gaussian_filter(x, sigma=(0, 1, 1, 0)),
    'grayscale': lambda x: np.repeat(x.mean(-1, keepdims=True), 3, -1),
}
xs = x_test.astype(np.float32)
per_class_acc = lambda yh: np.array([(yh[y_test == i] == i).mean() for i in range(10)])

rob_rows, rob_cls = [], {}
for name, f in perturb.items():
    yh = predict(f(xs)).argmax(1); a = per_class_acc(yh); rob_cls[name] = a
    rob_rows.append({'perturbation': name, 'overall acc': (yh == y_test).mean(), 'worst class': class_names[a.argmin()],
                     'worst-class acc': a.min(), 'class gap (max-min)': a.max() - a.min()})
rob_df = pd.DataFrame(rob_rows).round(4); display(rob_df)
RESULTS['robustness'] = rob_df.to_dict('records')

display(pd.DataFrame(rob_cls, index=class_names).round(3))

## 4. Global feature importance (permutation / occlusion)

Model-agnostic importance measured as the accuracy drop when a feature group is destroyed:
* **Colour channels:** permute R, G or B across images. A big drop = reliance on colour.
* **Spatial regions:** replace the centre 16x16 (the object, usually) or the border (background) with the dataset mean colour. A large *border* drop for a class hints at **background shortcuts** (e.g. sky for airplane, water for ship) rather than the object itself.

In [ ]:
base_acc = correct.mean()
ch_rows = []
for c, nm in enumerate('RGB'):
    xp = xs.copy(); xp[..., c] = xs[rng.permutation(len(xs))][..., c]
    ch_rows.append({'channel': nm, 'accuracy drop': base_acc - (predict(xp).argmax(1) == y_test).mean()})
ch_df = pd.DataFrame(ch_rows).round(4); display(ch_df)
RESULTS['channel_importance'] = ch_df.to_dict('records')

mean_pix = x_train.mean(axis=(0, 1, 2)).astype(np.float32)
def occlude(x, region):
    x = x.copy(); m = np.zeros(x.shape[1:3], bool)
    if region == 'centre': m[8:24, 8:24] = True
    else: m[:] = True; m[8:24, 8:24] = False
    x[:, m, :] = mean_pix; return x

pc_base = per_class_acc(y_pred)
drop = pd.DataFrame({
    'drop if centre hidden': pc_base - per_class_acc(predict(occlude(xs, 'centre')).argmax(1)),
    'drop if border hidden': pc_base - per_class_acc(predict(occlude(xs, 'border')).argmax(1)),
}, index=class_names).round(3)
drop['border/centre ratio'] = (drop['drop if border hidden'] / drop['drop if centre hidden'].clip(lower=1e-3)).round(2)
display(drop)
RESULTS['region_importance'] = drop.reset_index().rename(columns={'index': 'class'}).to_dict('records')

drop[['drop if centre hidden', 'drop if border hidden']].plot.bar(figsize=(9, 4), title='Accuracy drop when region is hidden')
plt.ylabel('Accuracy drop'); plt.tight_layout(); plt.show()

## 5. Local explanations with SHAP

SHAP (Partition explainer with a blur masker) assigns each pixel region a contribution to the predicted class. Red = pushes the prediction towards the class, blue = pushes away. We explain (a) correctly classified images and (b) **misclassified** images from the most-confused class pair, to see *why* the model fails.

In [ ]:
masker = shap.maskers.Image('blur(4,4)', x_test[0].shape)
shap_explainer = shap.Explainer(lambda x: predict(x), masker, output_names=class_names)

_, a, b = RESULTS['top_confusions'][0]['count'], RESULTS['top_confusions'][0]['true'], RESULTS['top_confusions'][0]['predicted']
ia, ib = class_names.index(a), class_names.index(b)
idx_right = np.where((y_test == ia) & correct)[0][:3]
idx_wrong = np.where((y_test == ia) & (y_pred == ib))[0][:3]
print(f'Most confused pair: true={a} predicted={b}')

for title, idx in [(f'Correctly classified ({a})', idx_right), (f'MISclassified: true {a}, predicted {b}', idx_wrong)]:
    print(title, '| predictions:', [class_names[y_pred[i]] + f' ({conf[i]:.2f})' for i in idx])
    sv = shap_explainer(x_test[idx].astype(np.float32), max_evals=500, batch_size=50, outputs=shap.Explanation.argsort.flip[:2])
    shap.image_plot(sv, show=True)

In [ ]:
# Aggregate SHAP: where does the evidence for each class sit spatially?  (8 correct images per class)
N_PER_CLASS = 8
agg, centre_share = [], []
for c in range(10):
    idx = np.where((y_test == c) & correct)[0][:N_PER_CLASS]
    sv = shap_explainer(x_test[idx].astype(np.float32), max_evals=300, batch_size=50, outputs=[c])
    m = np.abs(sv.values[..., 0]).sum(-1).mean(0)          # (32, 32) mean |SHAP|
    agg.append(m); centre_share.append(m[8:24, 8:24].sum() / m.sum())

fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
for c, ax in enumerate(axes.ravel()):
    ax.imshow(agg[c], cmap='hot'); ax.set_title(f'{class_names[c]}\ncentre share {centre_share[c]:.2f}'); ax.axis('off')
plt.suptitle('Mean |SHAP| per class (bright = more influence). Uniform centre share = 0.25'); plt.tight_layout(); plt.show()
RESULTS['shap_centre_share'] = {class_names[c]: round(float(centre_share[c]), 3) for c in range(10)}

## 6. Local explanations with LIME

LIME perturbs superpixels and fits a local linear model. Green outlines mark the superpixels that most support the predicted class. Comparing LIME with SHAP on the same images is a sanity check: if two methods disagree strongly, treat both explanations with caution.

In [ ]:
lime_exp = lime_image.LimeImageExplainer(random_state=0)
seg = lambda im: slic(im / 255.0, n_segments=20, compactness=10, start_label=1)

def lime_view(i, num_samples=1000):
    e = lime_exp.explain_instance(x_test[i].astype('double'), predict, top_labels=2, hide_color=0,
                                  num_samples=num_samples, batch_size=128, segmentation_fn=seg)
    lab = e.top_labels[0]
    temp, mask = e.get_image_and_mask(lab, positive_only=True, num_features=4, hide_rest=False)
    return mark_boundaries(temp / 255.0, mask), class_names[lab]

sel = list(idx_right) + list(idx_wrong)
fig, axes = plt.subplots(2, len(sel), figsize=(3 * len(sel), 6.2))
for k, i in enumerate(sel):
    axes[0, k].imshow(x_test[i]); axes[0, k].axis('off')
    axes[0, k].set_title(f'true: {class_names[y_test[i]]}\npred: {class_names[y_pred[i]]} ({conf[i]:.2f})', fontsize=9)
    img, lab = lime_view(i); axes[1, k].imshow(img); axes[1, k].axis('off'); axes[1, k].set_title(f'LIME evidence for: {lab}', fontsize=9)
plt.tight_layout(); plt.show()

## 7. Mitigation experiment (optional): class-reweighted fine-tuning

One practical mitigation for class-level disparity is to up-weight under-served classes. Weights come from the *training set's* per-class error (never the test set, to avoid leakage). The training set was seen during training, so these error rates are optimistic, but they still rank the hard classes. Set `RUN_MITIGATION = True` to run (about 5 minutes on CPU).

In [ ]:
RUN_MITIGATION = False

if RUN_MITIGATION:
    tr_pred = predict(x_train[:10000]).argmax(1); tr_y = y_train[:10000]
    tr_recall = np.array([(tr_pred[tr_y == c] == c).mean() for c in range(10)])
    w = 1.0 / np.clip(tr_recall, 0.3, 1.0); w = w / w.mean()
    class_weight = {c: float(w[c]) for c in range(10)}
    print('class weights:', {class_names[c]: round(v, 2) for c, v in class_weight.items()})

    m2 = tf.keras.models.clone_model(model); m2.set_weights(model.get_weights())
    m2.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    m2.fit(x_train, y_train, epochs=3, batch_size=64, class_weight=class_weight, validation_data=(x_test, y_test))

    p2 = m2.predict(x_test, batch_size=256, verbose=0).argmax(1)
    rec2 = per_class_acc(p2)
    mit = pd.DataFrame({'recall before': rec.values, 'recall after': rec2}, index=class_names).round(3)
    display(mit)
    summary = {'overall acc before': correct.mean(), 'overall acc after': (p2 == y_test).mean(),
               'recall gap before': rec.max() - rec.min(), 'recall gap after': rec2.max() - rec2.min(),
               'worst-class recall before': rec.min(), 'worst-class recall after': rec2.min()}
    display(pd.Series(summary).round(4)); RESULTS['mitigation'] = {k: round(float(v), 4) for k, v in summary.items()}
    mit[['recall before', 'recall after']].plot.bar(figsize=(9, 4), title='Per-class recall: baseline vs reweighted'); plt.tight_layout(); plt.show()
else:
    print('Skipped. Set RUN_MITIGATION = True to run the reweighting experiment.')

In [ ]:
# Write all measured numbers to a markdown file to paste into the write-up (REPORT.md).
def block(df): return '```\n' + df.to_string() + '\n```\n'
out = ['# Measured results (auto-generated)\n',
       f"Overall test accuracy: {RESULTS['overall_accuracy']*100:.2f}%\n",
       f"Recall gap (max-min): {RESULTS['recall_gap']:.3f}; min/max ratio: {RESULTS['recall_ratio_min_max']:.3f}; "
       f"worst class: {RESULTS['worst_class']}, best class: {RESULTS['best_class']}\n",
       '## Per-class metrics\n', block(class_df), '## Top confusions\n', block(pairs_df),
       '## Accuracy by condition slice\n', block(slice_df), '## Robustness\n', block(rob_df),
       '## Region importance (accuracy drop)\n', block(drop), '## Channel importance\n', block(ch_df),
       '## SHAP centre share per class (uniform = 0.25)\n', '```\n' + str(RESULTS['shap_centre_share']) + '\n```\n']
if 'mitigation' in RESULTS: out += ['## Mitigation experiment\n', '```\n' + str(RESULTS['mitigation']) + '\n```\n']
open('fairness_results.md', 'w').write('\n'.join(out))
print('Wrote fairness_results.md')